# 06 - AMP Reward Flow

本节目标: 你能解释 `task_reward`, `style_reward`, `final_reward`, `mixed_advantage` 的关系。

In [ ]:
import torch

torch.manual_seed(4)
N = 5
K = 4

env_group_rewards = torch.randn(N, K)
style_reward = torch.rand(N)
task_reward_lerp = 0.6

task_reward = env_group_rewards.sum(dim=-1)
final_reward = task_reward_lerp * task_reward + (1.0 - task_reward_lerp) * style_reward

print('env_group_rewards:', env_group_rewards.shape)
print('task_reward:', task_reward.shape)
print('style_reward:', style_reward.shape)
print('final_reward:', final_reward.shape)

## AMP 和 multi-critic 的边界

multi-critic 的 reward groups 是环境 reward 分解。

AMP 的 style reward 来自 discriminator。

在 multi-critic + AMP 里, `task_reward` 是所有环境 reward groups 的总和, 不是某一个 critic 分支。

In [ ]:
T = 4
N = 3
gamma = 0.99
lam = 0.95

env_weighted_adv = torch.randn(T, N)
style_rewards_rollout = torch.rand(T, N)
dones = torch.zeros(T, N)

style_advantages = torch.zeros(T, N)
style_adv = torch.zeros(N)
for step in reversed(range(T)):
    not_done = 1.0 - dones[step]
    style_adv = style_rewards_rollout[step] + not_done * gamma * lam * style_adv
    style_advantages[step] = style_adv

mixed_advantage = task_reward_lerp * env_weighted_adv + (1.0 - task_reward_lerp) * style_advantages

print('env_weighted_adv:', env_weighted_adv.shape)
print('style_advantages:', style_advantages.shape)
print('mixed_advantage:', mixed_advantage.shape)

## 作业

1. 解释为什么 AMP discriminator 不是 critic head。
2. 把 `task_reward_lerp=1.0`, 看 mixed advantage 退化成什么。
3. 把 `task_reward_lerp=0.0`, 看 actor 只受哪条 reward 影响。
4. 用自己的话写出 `AMPPPO.process_env_step` 应该负责什么。